# Part A — Model Derivation and Linearisation

**ES4G4 Nonlinear Control and Reinforcement Learning · Assignment 1 (Python rebuild)**

This notebook derives the nonlinear state-space model of the quadrotor attitude
dynamics (A-1), linearises it about the hover equilibrium, reports the parametric
and numeric $(\mathbf{A}, \mathbf{B})$ matrices, and analyses stability and
controllability (A-2).

Everything symbolic is done with `sympy` so the Jacobians are **computed, not
transcribed** — the numeric matrices then follow by mechanical substitution of
the Table 1 values, with zero copy-error risk.

In [2]:
import sys
print(sys.executable)

c:\Users\User\Desktop\Warwick Lectures\Reports\Quadrotor Attitude\.venv\Scripts\python.exe


In [ ]:
import sympy as sp
import numpy as np
from IPython.display import display, Math

sp.init_printing(use_latex="mathjax")
np.set_printoptions(precision=4, suppress=True, linewidth=120)

## A-1 · State-space representation

### Choice of state and input vectors

The governing equations (brief, p.4) are six coupled first-order ODEs in six
dependent variables. The **minimal** realisation therefore takes exactly those
six as states — three attitude angles and three body-frame rates — with the
three rotor torques as inputs:

$$
\mathbf{x} = \begin{bmatrix} \phi & \theta & \psi & \omega_x & \omega_y & \omega_z \end{bmatrix}^{\!T}\!, \qquad
\mathbf{u} = \begin{bmatrix} \tau_x & \tau_y & \tau_z \end{bmatrix}^{\!T}
$$

so $n = 6$, $m = 3$. Minimality is immediate: no state can be removed (each
appears dynamically coupled to the others through the kinematics or Euler's
equations), and no extra states are introduced.

**Ordering rationale.** Angles first, rates second. This is deliberate: after
linearisation it produces a clean block-triangular
$\mathbf{A} = \begin{bmatrix} \mathbf{0} & \mathbf{I}_3 \\ \mathbf{0} & \mathbf{D} \end{bmatrix}$
structure (kinematic integrators on top, damped rate dynamics below) whose
eigenvalues are readable by inspection.

In [12]:
# Symbols
phi, theta, psi = sp.symbols(r'phi theta psi', real=True)
wx, wy, wz = sp.symbols(r'omega_x omega_y omega_z', real=True)
tx, ty, tz = sp.symbols(r'tau_x tau_y tau_z', real=True)
Ix, Iy, Iz = sp.symbols(r'I_x I_y I_z', positive=True)
dx, dy, dz = sp.symbols(r'd_x d_y d_z', positive=True)

x = sp.Matrix([phi, theta, psi, wx, wy, wz])
u = sp.Matrix([tx, ty, tz])

display(Math(r"\mathbf{x} = " + sp.latex(x.T) + r"^T, \qquad \mathbf{u} = "
             + sp.latex(u.T) + r"^T"))

<IPython.core.display.Math object>

### Nonlinear state equations

Rearranging the given dynamics into explicit first-order form
$\dot{\mathbf{x}} = f(\mathbf{x}, \mathbf{u})$: the three kinematic equations
are already explicit in $(\dot\phi, \dot\theta, \dot\psi)$, and Euler's
equations only need division by the (positive, hence invertible) inertias.
Note every $f_i$ depends **only on $\mathbf{x}$ and $\mathbf{u}$**, as required.

In [5]:
f = sp.Matrix([
    # kinematics: Euler-angle rates driven by body rates
    wx + sp.sin(phi)*sp.tan(theta)*wy + sp.cos(phi)*sp.tan(theta)*wz,
    sp.cos(phi)*wy - sp.sin(phi)*wz,
    (sp.sin(phi)/sp.cos(theta))*wy + (sp.cos(phi)/sp.cos(theta))*wz,
    # Euler's equations: gyroscopic coupling + linear aerodynamic damping
    (tx + (Iy - Iz)*wy*wz - dx*wx) / Ix,
    (ty + (Iz - Ix)*wz*wx - dy*wy) / Iy,
    (tz + (Ix - Iy)*wx*wy - dz*wz) / Iz,
])

for i in range(6):
    display(Math(rf"\dot{{x}}_{{{i+1}}} = f_{{{i+1}}}(\mathbf{{x}},\mathbf{{u}}) = "
                 + sp.latex(f[i])))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## A-2 · Linearisation about hover

### The equilibrium must be verified, not assumed

Jacobian linearisation about $(\mathbf{x}^*, \mathbf{u}^*)$ is the first-order
truncation of the Taylor expansion

$$
\dot{\mathbf{x}} \;\approx\; \underbrace{f(\mathbf{x}^*, \mathbf{u}^*)}_{\text{must be } \mathbf{0}}
\;+\; \underbrace{\left.\frac{\partial f}{\partial \mathbf{x}}\right|_{(\mathbf{x}^*,\mathbf{u}^*)}}_{\mathbf{A}} \delta\mathbf{x}
\;+\; \underbrace{\left.\frac{\partial f}{\partial \mathbf{u}}\right|_{(\mathbf{x}^*,\mathbf{u}^*)}}_{\mathbf{B}} \delta\mathbf{u}
$$

The expansion only yields an LTI model if the constant term vanishes, i.e. if
$(\mathbf{x}^*, \mathbf{u}^*)$ is genuinely an equilibrium — so we check
$f(\mathbf{0}, \mathbf{0}) = \mathbf{0}$ explicitly before proceeding.

In [7]:
eq = {s: 0 for s in [phi, theta, psi, wx, wy, wz, tx, ty, tz]}
f_at_eq = f.subs(eq)
assert f_at_eq == sp.zeros(6, 1)
display(Math(r"f(\mathbf{0}, \mathbf{0}) = " + sp.latex(f_at_eq.T)
             + r"^T \;\;\Rightarrow\;\; \text{hover is an equilibrium} \;\checkmark"))

<IPython.core.display.Math object>

### What the evaluation at hover kills

Two families of terms vanish at $(\mathbf{x}^*, \mathbf{u}^*) = (\mathbf{0}, \mathbf{0})$:

1. **Trigonometric couplings** in $f_1$–$f_3$: $\sin 0 = \tan 0 = 0$,
   $\cos 0 = 1$, so the kinematics collapse to $\dot\phi \approx \omega_x$,
   $\dot\theta \approx \omega_y$, $\dot\psi \approx \omega_z$.
2. **Bilinear gyroscopic terms** $\omega_i \omega_j$ in $f_4$–$f_6$: each
   partial derivative $\partial(\omega_y\omega_z)/\partial\omega_y = \omega_z$
   etc. evaluates to zero at the origin.

The consequence — worth flagging because it becomes the central plot point of
Parts C and D — is that the linearised model contains **no cross-axis coupling
whatsoever**: roll, pitch and yaw decouple into three independent double-
integrator-with-damping channels. All coupling in the real plant is second
order and invisible to this model.

In [8]:
A_sym = f.jacobian(x).subs(eq)
B_sym = f.jacobian(u).subs(eq)

display(Math(r"\mathbf{A} = \left.\frac{\partial f}{\partial \mathbf{x}}\right|_{(\mathbf{0},\mathbf{0})} = "
             + sp.latex(A_sym)))
display(Math(r"\mathbf{B} = \left.\frac{\partial f}{\partial \mathbf{u}}\right|_{(\mathbf{0},\mathbf{0})} = "
             + sp.latex(B_sym)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Numeric matrices (Table 1 nominal values)

$$
\frac{d_x}{I_x} = \frac{d_y}{I_y} = \frac{10^{-5}}{6\times 10^{-5}} = \frac{1}{6}, \qquad
\frac{d_z}{I_z} = \frac{2\times 10^{-5}}{1.2\times 10^{-4}} = \frac{1}{6}
$$

In [9]:
params = {Ix: sp.Rational(6, 100000), Iy: sp.Rational(6, 100000),
          Iz: sp.Rational(12, 100000),
          dx: sp.Rational(1, 100000), dy: sp.Rational(1, 100000),
          dz: sp.Rational(2, 100000)}

A_exact = A_sym.subs(params)   # keep exact rationals for display
B_exact = B_sym.subs(params)
display(Math(r"\mathbf{A} = " + sp.latex(A_exact)))
display(Math(r"\mathbf{B} = " + sp.latex(B_exact)))

A_num = np.array(A_exact, dtype=float)
B_num = np.array(B_exact, dtype=float)
print("A =\n", A_num, "\n\nB =\n", B_num)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

A =
 [[ 0.      0.      0.      1.      0.      0.    ]
 [ 0.      0.      0.      0.      1.      0.    ]
 [ 0.      0.      0.      0.      0.      1.    ]
 [ 0.      0.      0.     -0.1667  0.      0.    ]
 [ 0.      0.      0.      0.     -0.1667  0.    ]
 [ 0.      0.      0.      0.      0.     -0.1667]] 

B =
 [[    0.         0.         0.    ]
 [    0.         0.         0.    ]
 [    0.         0.         0.    ]
 [16666.6667     0.         0.    ]
 [    0.     16666.6667     0.    ]
 [    0.         0.      8333.3333]]


### Cross-check against the shared simulation module

`quadrotor.py` is the single source of truth used by Parts B–D. We verify the
hard-coded matrices there agree with the symbolic derivation here, and — as a
stronger consistency check — that the Jacobian of the *numeric* nonlinear
dynamics (finite differences at the origin) matches too. This catches any
transcription error between the symbolic model and the simulation code.

In [11]:
from quadrotor import linearised_matrices, f_nonlinear

A_mod, B_mod = linearised_matrices()
assert np.allclose(A_mod, A_num) and np.allclose(B_mod, B_num)

# finite-difference Jacobian of the numeric f at the origin
eps = 1e-7
A_fd = np.zeros((6, 6)); B_fd = np.zeros((6, 3))
x0, u0 = np.zeros(6), np.zeros(3)
for j in range(6):
    dxv = np.zeros(6); dxv[j] = eps
    A_fd[:, j] = (f_nonlinear(x0 + dxv, u0) - f_nonlinear(x0 - dxv, u0)) / (2*eps)
for j in range(3):
    duv = np.zeros(3); duv[j] = eps
    B_fd[:, j] = (f_nonlinear(x0, u0 + duv) - f_nonlinear(x0, u0 - duv)) / (2*eps)

assert np.allclose(A_fd, A_num, atol=1e-5)
assert np.allclose(B_fd, B_num, rtol=1e-6)
print("Symbolic Jacobian == quadrotor.py matrices == finite-difference "
      "Jacobian of the numeric nonlinear model  ✓")

Symbolic Jacobian == quadrotor.py matrices == finite-difference Jacobian of the numeric nonlinear model  ✓


## Stability of the linearised plant

The eigenvalues of $\mathbf{A}$ follow by inspection from its block-triangular
structure: the spectrum is the union of the spectra of the diagonal blocks,
$\{0, 0, 0\}$ from the integrator block and $\{-\tfrac16, -\tfrac16, -\tfrac16\}$
from the damping block.

In [13]:
eigvals = np.linalg.eigvals(A_num)
print("eigenvalues of A:", np.sort(eigvals.real))

# Jordan structure of the repeated zero eigenvalue: marginal stability requires
# lambda = 0 to be NON-DEFECTIVE (geometric multiplicity == algebraic).
rank_A = np.linalg.matrix_rank(A_num)
geo_mult = 6 - rank_A
print(f"algebraic multiplicity of λ=0 : 3")
print(f"geometric multiplicity of λ=0 : {geo_mult}  (dim null(A))")
assert geo_mult == 3

eigenvalues of A: [-0.1667 -0.1667 -0.1667  0.      0.      0.    ]
algebraic multiplicity of λ=0 : 3
geometric multiplicity of λ=0 : 3  (dim null(A))


### Interpretation — stated precisely

Three separate claims, each with its own justification. Conflating them is a
common error (and cost marks in the original report):

1. **The rate subsystem is asymptotically stable.** $\lambda_{4,5,6} = -\tfrac16$:
   left half-plane, so the body rates decay under aerodynamic damping alone.

2. **The linearised system is marginally (Lyapunov) stable — not asymptotically
   stable, and not unstable.** The repeated eigenvalue $\lambda = 0$ has
   geometric multiplicity 3 equal to its algebraic multiplicity (verified
   above), so there are **no Jordan blocks** and hence no polynomially growing
   modes. Angle perturbations persist indefinitely but do not grow. Had
   $\lambda = 0$ been defective — as it is, e.g., in a double integrator —
   the linear system would be Lyapunov-*unstable* despite having no eigenvalue
   in the open right half-plane, which is why this check is not a formality.

3. **The linearisation is *inconclusive* about the nonlinear hover
   equilibrium.** Lyapunov's indirect method decides stability only when no
   eigenvalue lies on the imaginary axis. With three eigenvalues *at* zero this
   is the critical case: the discarded higher-order terms determine the
   verdict, and no claim that "hover is unstable" (or stable) may be made from
   $\mathbf{A}$ alone. What *can* be said: the linearisation offers no
   asymptotic stability, so active feedback is required to obtain it — which
   is precisely the control problem posed in Part B.

## Controllability

Existence of a stabilising controller must be established, not assumed. With
$n = 6$ we form the controllability matrix
$\mathcal{C} = [\,\mathbf{B} \;\; \mathbf{AB} \;\; \cdots \;\; \mathbf{A}^{5}\mathbf{B}\,]$
and check its rank.

In [14]:
C = np.hstack([np.linalg.matrix_power(A_num, k) @ B_num for k in range(6)])
r = np.linalg.matrix_rank(C)
print(f"rank(C) = {r} of {6}")
assert r == 6

rank(C) = 6 of 6


$\mathrm{rank}(\mathcal{C}) = 6$: the pair
$(\mathbf{A}, \mathbf{B})$ is **fully controllable**. All six states — in
particular the three marginally stable integrator angle states — can be driven
arbitrarily by the three torques, so pole placement (and a fortiori a
stabilising MPC) is feasible. Intuitively this is clear from the structure:
each torque directly accelerates one body rate, and each rate directly
integrates into one angle.

## Summary

| Item | Result |
|---|---|
| States / inputs | $n = 6$, $m = 3$; minimal realisation |
| Equilibrium | $f(\mathbf{0},\mathbf{0}) = \mathbf{0}$ verified |
| Linearised structure | three decoupled axes; all coupling is 2nd-order |
| Eigenvalues | $\{0,0,0,\; -\tfrac16,-\tfrac16,-\tfrac16\}$ |
| Linear stability | marginally stable ($\lambda{=}0$ non-defective) |
| Nonlinear equilibrium | indirect method inconclusive (critical case) |
| Controllability | full rank — stabilising feedback exists |

Next: **Part B**, where the controllable-but-only-marginally-stable linear
model becomes the internal prediction model of a constrained MPC.